In [1]:
!pip install transformers sentencepiece scikit-learn --quiet

In [29]:
import os
import torch
import warnings
import logging
import numpy as np
import pandas as pd
from transformers import (
    pipeline,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    BertTokenizer, BertForSequenceClassification
)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from google.colab import drive
import time
from tqdm import tqdm

In [16]:
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger().setLevel(logging.ERROR)

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

drive.mount('/content/drive', force_remount=True)

drive_path = '/content/drive/MyDrive/Event Log/2025-7-28 申毅实习/策略研究/NLP 日度'
os.chdir(drive_path)

Mounted at /content/drive


In [17]:
device = 0 if torch.cuda.is_available() else -1
print("Device set to", "GPU" if device == 0 else "CPU")

Device set to GPU


In [39]:
nllb_model_name = "facebook/nllb-200-1.3B"
nllb_tokenizer = AutoTokenizer.from_pretrained(nllb_model_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_model_name)

translator = pipeline(
    "translation",
    model=nllb_model,
    tokenizer = nllb_tokenizer,
    src_lang = "zho_Hans",
    tgt_lang = "eng_Latn",
    device=device,
    truncation = True,
    max_length = 512
)

In [20]:
finbert_model = "yiyanghkust/finbert-tone"
tokenizer = BertTokenizer.from_pretrained(finbert_model)
model = BertForSequenceClassification.from_pretrained(finbert_model)

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model = model,
    tokenizer = tokenizer,
    device = device
)

In [6]:
equity_reports = pd.read_csv(
    'equity_reports.csv',
    encoding='utf-8',
    parse_dates=['PUBLISH_DATE', 'UPDATE_TIME', 'UPDATE_DATE']
)

In [40]:
sentiment_map = {
    'POSITIVE': '正面',
    'NEUTRAL': '中性',
    'NEGATIVE': '负面'
}

verbose = True

In [42]:
start_time = time.time()

for i, row in equity_reports.head(10).iterrows():
    if verbose:
        print(f"\n=== Row {i + 1} ===")

    title_cn = row['TITLE']
    summary_cn = row['SUMMARY']

    try:
        if pd.notna(title_cn):
            title_en = translator(title_cn, max_length=512, truncation=True)[0]['translation_text']
            title_sentiment = sentiment_pipeline(title_en)[0]
            title_label = sentiment_map.get(title_sentiment['label'].upper(), '[未知]')
            if verbose:
                print(f"TITLE: {title_cn[:50]}{'...' if len(title_cn) > 50 else ''}")
                print(f"  ↳ {title_en[:100]}{'...' if len(title_en) > 100 else ''}")
                print(f"  → 情绪: {title_label} ({title_sentiment['score'] * 100:.1f}%)")
        else:
            if verbose:
                print("TITLE: [缺失]")

        if pd.notna(summary_cn):
            summary_en = translator(summary_cn, max_length=512, truncation=True)[0]['translation_text']
            summary_sentiment = sentiment_pipeline(summary_en)[0]
            summary_label = sentiment_map.get(summary_sentiment['label'].upper(), '[未知]')
            if verbose:
                print(f"SUMMARY: {summary_cn[:50]}{'...' if len(summary_cn) > 50 else ''}")
                print(f"  ↳ {summary_en[:100]}{'...' if len(summary_en) > 100 else ''}")
                print(f"  → 情绪: {summary_label} ({summary_sentiment['score'] * 100:.1f}%)")
        else:
            if verbose:
                print("SUMMARY: [缺失]")

    except Exception as e:
        print(f"Error on row {i}: {e}")

end_time = time.time()


=== Row 1 ===
TITLE: 诺诚健华-B（09969）：奥布替尼两项淋巴瘤适应症获批，纳入港股通期待估值修复
  ↳ No Chen-B ((09969)): Obteni has two lymphoma adaptations approved for inclusion in the expected reva...
  → 情绪: 中性 (100.0%)
SUMMARY:     投资要点事件：近日，（1）国家药监局批准公司的1类创新药奥布替尼（商品名：宜诺凯）上市，用于...
  ↳ Investment highlights: ((1) The National Pharmaceutical Administration approved the company's Class ...
  → 情绪: 正面 (99.2%)

=== Row 2 ===
TITLE: 金山办公（688111）公司深度报告之二：循成长之迹，探WPS发展远景
  ↳ The second in-depth report: Growth trajectory and prospects for the development of WPS
  → 情绪: 中性 (93.5%)
SUMMARY: 战略明确，长期成长路径清晰，维持“买入”评级无论是中端期，还是长期视角看，公司均表现出优异的成长性。...
  ↳ Strategic clarity, long-term growth path clear, maintaining the company's R&D ratings both in the me...
  → 情绪: 正面 (100.0%)

=== Row 3 ===
TITLE: 共创草坪（605099）：股权激励划定明确目标，人造草坪龙头再添增长动力
  ↳ Shareholder incentives to set clear targets, artificial turf to drive growth
  → 情绪: 正面 (99.8%)
Error on row 2: The size of tensor a (918) must match the size of tensor b (512) at non-singleton dim

In [31]:
elapsed = end_time - start_time
print(f"\nProcessed 100 rows in {elapsed:.2f} seconds")
print(f"Average time per row: {elapsed / 100:.2f} seconds")


Processed 100 rows in 869.29 seconds
Average time per row: 8.69 seconds


In [36]:
count = (equity_reports['Lag'] < 6).sum()
print(f"Rows where Lag: {count}")

Rows where Lag: 301430


In [37]:
len(equity_reports)

324452